In [13]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification, get_linear_schedule_with_warmup
import pandas as pd
import librosa
import os
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.nn import CrossEntropyLoss
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import numpy as np

In [14]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=7,
    ignore_mismatched_sizes=True
)

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 26025.53it/s]
[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_q.bias               | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
projector.bias               | MISSING    | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 
projector.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
df_train = pd.read_csv("C:/MELD.Raw/train/train_sent_emo.csv")
df_test = pd.read_csv("C:/MELD.Raw/test/test_sent_emo.csv")
df_dev = pd.read_csv("C:/MELD.Raw/dev/dev_sent_emo.csv")

label_map = {
    "neutral": 0, "joy": 1, "sadness": 2,
    "anger": 3, "fear": 4, "surprise": 5, "disgust": 6
}
df_train['label'] = df_train['Emotion'].map(label_map)
df_test['label'] = df_test['Emotion'].map(label_map)
df_dev['label'] = df_dev['Emotion'].map(label_map)
df_dev['label'] = df_dev['Emotion'].map(label_map)

In [16]:
def load_audio(file_path):
    try:
        y, sr = librosa.load(file_path, sr=16000)
        if len(y) == 0:
            return np.zeros(16000, dtype=np.float32)
        return y.astype(np.float32)
    except:
        return np.zeros(16000, dtype=np.float32)

In [17]:
class AudioDataset(Dataset):
    def __init__(self, df, audio_folder):
        self.df = df.reset_index(drop=True)
        self.audio_folder = audio_folder

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dia_id = row['Dialogue_ID']
        utt_id = row['Utterance_ID']
        filename = f"dia{dia_id}_utt{utt_id}.wav"
        full_path = os.path.join(self.audio_folder, filename)

        audio = load_audio(full_path) if os.path.exists(full_path) else np.zeros(48000, dtype=np.float32)

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            padding="max_length",
            max_length=48000,
            truncation=True
        )

        return {
            "input_values": inputs['input_values'].squeeze(0),
            "labels": torch.tensor(row['label'], dtype=torch.long)
        }

In [18]:
train_dataset = AudioDataset(df_train, "C:/MELD.Raw/Audio/train")
dev_dataset = AudioDataset(df_dev, "C:/MELD.Raw/Audio/dev")
test_dataset = AudioDataset(df_test, "C:/MELD.Raw/Audio/test")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [19]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Using device: {device}")
print(model)

Using device: cuda
Wav2Vec2ForSequenceClassification(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0

In [20]:
for param in model.wav2vec2.parameters():
    param.requires_grad = True

In [21]:
y = df_train['label'].values
class_weight = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0,1,2,3,4,5,6]),
    y=y
)
class_weight_tensor = torch.tensor(class_weight, dtype=torch.float).to(device)
loss_fn = CrossEntropyLoss(weight=class_weight_tensor)

In [ ]:
total_steps = len(train_loader) * 15
warmup_steps = total_steps // 15
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Training loop
for epoch in range(15):
    print(f"Epoch {epoch+1}/15")
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_values = batch["input_values"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_values=input_values)
        logits = outputs.logits
        loss = loss_fn(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Average Loss: {avg_loss:.4f}")


Epoch 1/15


In [ ]:
# model.save_pretrained("wav2vec_model")
# processor.save_pretrained("wav2vec_model")
# print("Model saved!")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.52it/s]

Model saved!


In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_values = batch["input_values"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_values=input_values)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

target_names = ['neutral', 'joy', 'sadness', 'anger', 'fear', 'surprise', 'disgust']
print(classification_report(all_labels, all_preds, target_names=target_names))

              precision    recall  f1-score   support

     neutral       0.60      0.49      0.54      1256
         joy       0.28      0.32      0.29       402
     sadness       0.19      0.35      0.25       208
       anger       0.41      0.28      0.33       345
        fear       0.02      0.04      0.03        50
    surprise       0.27      0.36      0.31       281
     disgust       0.04      0.03      0.03        68

    accuracy                           0.39      2610
   macro avg       0.26      0.27      0.26      2610
weighted avg       0.43      0.39      0.40      2610

